# VogueVision### AI-Powered Fashion Intelligence System### DS/ML project using H&M Personalized Fashion Recommendations dataset**4 Modules:**1. Price Prediction (Regression)2. Recommendation System (Content-based)3. Image-based Style Classification (CNN)4. Trend/Demand Forecasting (Time Series)---### Step 1: Get the datasetGo to https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/data1. Create a free Kaggle account if you don't have one2. Go to Kaggle Account Settings -> API -> "Create New API Token" -> downloads `kaggle.json`3. Upload `kaggle.json` when prompted below

In [ ]:
from google.colab import filesuploaded = files.upload()  # upload kaggle.json here!mkdir -p ~/.kaggle!cp kaggle.json ~/.kaggle/!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!pip install kaggle -q!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f articles.csv!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f customers.csv!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f transactions_train.csv!unzip -o articles.csv.zip!unzip -o customers.csv.zip!unzip -o transactions_train.csv.zip

**Note:** The full dataset is huge (31M+ transactions, 100k+ images).For a portfolio project, we'll work with a SAMPLE — this is normal practice and shows you know how to handle real-world scale data, not just toy datasets.

## Step 2: Load & Explore Data (EDA)

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsarticles = pd.read_csv('articles.csv')customers = pd.read_csv('customers.csv')# Transactions file is huge — read only a sample for this projecttransactions = pd.read_csv('transactions_train.csv', nrows=500000)print("Articles:", articles.shape)print("Customers:", customers.shape)print("Transactions (sample):", transactions.shape)articles.head()

In [ ]:
transactions.head()

In [ ]:
# Basic EDAprint(articles['product_group_name'].value_counts())plt.figure(figsize=(10,5))articles['product_group_name'].value_counts().head(10).plot(kind='bar')plt.title('Top Product Groups')plt.show()

In [ ]:
plt.figure(figsize=(8,5))sns.histplot(transactions['price'], bins=50)plt.title('Price Distribution')plt.show()print(transactions['price'].describe())

## Module 1: Price Prediction (Regression)Predict product price from article attributes (product type, department, color, etc.)

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import LabelEncoderfrom sklearn.ensemble import RandomForestRegressorfrom sklearn.metrics import mean_absolute_error, r2_score# Merge avg price per article from transactions with article featuresprice_df = transactions.groupby('article_id')['price'].mean().reset_index()price_data = price_df.merge(articles, on='article_id', how='left')features = ['product_group_name', 'colour_group_name', 'department_name',            'index_name', 'garment_group_name']price_data = price_data.dropna(subset=features + ['price'])X = price_data[features].copy()y = price_data['price']encoders = {}for col in features:    le = LabelEncoder()    X[col] = le.fit_transform(X[col].astype(str))    encoders[col] = leX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)model_price = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)model_price.fit(X_train, y_train)preds = model_price.predict(X_test)print("MAE:", mean_absolute_error(y_test, preds))print("R2 Score:", r2_score(y_test, preds))

In [ ]:
# Feature importanceimportances = pd.Series(model_price.feature_importances_, index=features).sort_values(ascending=False)importances.plot(kind='barh', title='Feature Importance - Price Prediction')plt.show()

## Module 2: Recommendation System (Content-Based)Recommend similar products based on product attributes using cosine similarity.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.metrics.pairwise import cosine_similarity# Use a sample of articles for similarity (full dataset is large)sample_articles = articles.sample(5000, random_state=42).reset_index(drop=True)# Combine text features into one string per productsample_articles['combined_features'] = (    sample_articles['product_group_name'].astype(str) + ' ' +    sample_articles['colour_group_name'].astype(str) + ' ' +    sample_articles['department_name'].astype(str) + ' ' +    sample_articles['garment_group_name'].astype(str) + ' ' +    sample_articles['detail_desc'].astype(str))tfidf = TfidfVectorizer(stop_words='english', max_features=2000)tfidf_matrix = tfidf.fit_transform(sample_articles['combined_features'])cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)def recommend_similar(article_idx, top_n=5):    sim_scores = list(enumerate(cosine_sim[article_idx]))    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]    indices = [i[0] for i in sim_scores]    return sample_articles.iloc[indices][['article_id', 'prod_name', 'product_group_name', 'colour_group_name']]# Example: recommend products similar to article at index 0print("Selected product:")print(sample_articles.iloc[0][['prod_name', 'product_group_name', 'colour_group_name']])print("\nRecommended similar products:")recommend_similar(0)

## Module 3: Image-Based Style Classification (CNN)Classify product images by category using transfer learning (MobileNetV2).Requires downloading a sample of product images.

In [ ]:
# Download a small sample of images for the classification task# The full image folder is huge, so this pulls a manageable subset via Kaggle API!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f images.zip# NOTE: images.zip is very large (~25GB). For a portfolio project on Colab,# it's more practical to download a curated subset manually from Kaggle's file browser,# or use the 'images' folder for ~500-1000 sample images matching your price_data article_ids.# Instructions: https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/data# -> browse to 'images/' folder -> download a few subfolders (organized by article_id prefix)

In [ ]:
import tensorflow as tffrom tensorflow.keras.applications import MobileNetV2from tensorflow.keras.applications.mobilenet_v2 import preprocess_inputfrom tensorflow.keras.preprocessing.image import ImageDataGeneratorfrom tensorflow.keras import layers, models# Once you have a local 'images/' folder organized as images/<class_name>/<image>.jpg# (you can create this structure by copying images into folders based on product_group_name)IMG_SIZE = (224, 224)BATCH_SIZE = 32datagen = ImageDataGenerator(    preprocessing_function=preprocess_input,    validation_split=0.2)train_gen = datagen.flow_from_directory(    'images/',    target_size=IMG_SIZE,    batch_size=BATCH_SIZE,    class_mode='categorical',    subset='training')val_gen = datagen.flow_from_directory(    'images/',    target_size=IMG_SIZE,    batch_size=BATCH_SIZE,    class_mode='categorical',    subset='validation')base_model = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')base_model.trainable = False  # freeze pretrained layersmodel_img = models.Sequential([    base_model,    layers.GlobalAveragePooling2D(),    layers.Dense(128, activation='relu'),    layers.Dropout(0.3),    layers.Dense(train_gen.num_classes, activation='softmax')])model_img.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])model_img.summary()

In [ ]:
history = model_img.fit(train_gen, validation_data=val_gen, epochs=10)plt.plot(history.history['accuracy'], label='train acc')plt.plot(history.history['val_accuracy'], label='val acc')plt.legend()plt.title('Style Classification Accuracy')plt.show()

## Module 4: Trend / Demand Forecasting (Time Series)Forecast daily sales volume for a product category using historical transaction data.

In [ ]:
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])merged = transactions.merge(articles[['article_id', 'product_group_name']], on='article_id', how='left')# Pick a top category to forecasttop_category = merged['product_group_name'].value_counts().idxmax()print("Forecasting demand for:", top_category)daily_sales = merged[merged['product_group_name'] == top_category].groupby('t_dat').size().reset_index(name='units_sold')daily_sales = daily_sales.set_index('t_dat').asfreq('D').fillna(0)plt.figure(figsize=(12,5))daily_sales['units_sold'].plot()plt.title(f'Daily Units Sold - {top_category}')plt.show()

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothingtrain_size = int(len(daily_sales) * 0.85)train_ts, test_ts = daily_sales.iloc[:train_size], daily_sales.iloc[train_size:]model_ts = ExponentialSmoothing(train_ts['units_sold'], trend='add', seasonal='add', seasonal_periods=7).fit()forecast = model_ts.forecast(len(test_ts))plt.figure(figsize=(12,5))plt.plot(train_ts.index, train_ts['units_sold'], label='Train')plt.plot(test_ts.index, test_ts['units_sold'], label='Actual')plt.plot(test_ts.index, forecast, label='Forecast')plt.legend()plt.title(f'Demand Forecast - {top_category}')plt.show()from sklearn.metrics import mean_absolute_errorprint("Forecast MAE:", mean_absolute_error(test_ts['units_sold'], forecast))

## SummaryThis project demonstrates 4 core DS/ML skills using one real-world retail dataset:- **Regression** (price prediction)- **Content-based recommendation** (similarity search)- **Computer vision / transfer learning** (image classification)- **Time-series forecasting** (demand prediction)### Next steps to strengthen this for interviews:1. Add a short write-up (README) explaining business context + your findings for each module2. Deploy one module (e.g. recommendation system) as a simple Streamlit app3. Push to GitHub with clear commit history and a README with screenshots4. Be ready to explain: why you picked this dataset, key challenges (data size, missing values), and what you'd improve with more time